# Direct original-plant delay simulations
This notebook uses physical X trajectories, never a target trajectory to generate the response.
2016: explicit sampled/ZOH predictor feedback with sample-period refinement.
2012: continuous-time functional feedback approximated with Heun and causal state interpolation.
The printed factor-1 and chain-rule factor-2 specializations are kept separate.
CPU only; no GPU is needed.

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ('numpy','scipy','matplotlib','pytest') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
from pathlib import Path
roots = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p/'python/src/direct_dde.py').exists()), None)
if root is None:
    root=Path('/content/delay-reproduction') if Path('/content').exists() else Path.cwd()/'delay-reproduction'
    if not root.exists():
        subprocess.run(['git','clone','--depth','1','--branch','reproducibility-audit-notes',
                        'https://github.com/KK1182112KK/krstic-2016-reproduction.git',str(root)],check=True)
    if not (root/'python/src/direct_dde.py').exists():
        raise RuntimeError('Checkout does not contain the direct-simulation revision.')
sys.path.insert(0,str(root/'python/src'))
sys.path.insert(0,str(root/'state-delay-2012'))
import numpy as np, scipy, matplotlib.pyplot as plt
from direct_dde import simulate, identity_residual
from direct_state_delay import simulate as simulate_state_delay, summarize
print('Python:',sys.version,'NumPy:',np.__version__,'SciPy:',scipy.__version__)

In [ ]:
subprocess.run([sys.executable,'-m','pytest',str(root/'python/tests'),'-q'],check=True)

## 2016 original input-delay plant
The same U acts immediately on X1 and with delay on X2. Only X is propagated.
U is computed at each sample from X and stored applied input, then held until the next sample.
The zero negative-time input and U(0+) are deliberately distinct.

In [ ]:
r=simulate(D=1.,sample_dt=.005,t_end=20.,X0=(1.,1.))
print('Final physical norm:',np.linalg.norm(r.X[-1]))
plt.figure()
plt.plot(r.t,r.X[:,0],label='X1'); plt.plot(r.t,r.X[:,1],label='X2')
plt.xlabel('Time'); plt.ylabel('Physical state'); plt.legend(); plt.show()
plt.figure()
plt.step(r.t,r.U,where='post',label='Applied U')
plt.xlabel('Time'); plt.ylabel('Held control'); plt.legend(); plt.show()

In [ ]:
subprocess.run([sys.executable,str(root/'python/run_direct_validation.py')],cwd=root,check=True)
print((root/'results/direct-dde/summary.json').read_text())

## 2012 original state-delay plant: printed vs chain-rule specialization
Both runs integrate s_dot=v(t-D(s)), v_dot=a with the same physical delay.
Factor 1 is the printed formula. Factor 2 is an explicitly labeled alternative.
No claim about the authors' source code or exact pixel agreement is made.

In [ ]:
state_runs={f:simulate_state_delay(factor=f,dt=.0025,t_end=6.) for f in (1,2)}
for f,sr in state_runs.items():
    print('factor',f,summarize(sr))
plt.figure()
for f,sr in state_runs.items():
    plt.plot(sr.t,sr.U,label=f'factor {f}')
plt.xlabel('Time'); plt.ylabel('Control a'); plt.legend(); plt.show()

## Cooling: stated history vs explicit alternative
T_eq=0.4 is a hypothesis used for this audit, not a value in the paper's parameter sentence.
The original plant and controller parameters otherwise remain as stated.

In [ ]:
cooling={h:simulate_state_delay(example='cooling',x2_history=h,Teq=.4,dt=.005,t_end=10.) for h in (.2,.6)}
plt.figure()
for h,sr in cooling.items():
    plt.plot(sr.t,sr.X[:,1],label=f'X2 history {h}')
    print('history',h,'U(0+)',sr.U[0],'final X',sr.X[-1])
plt.xlabel('Time'); plt.ylabel('Physical X2'); plt.legend(); plt.show()

## Limits
These are finite-horizon numerical computations, not theorem proofs, certified regions of attraction,
or actuator-robustness guarantees. Neither singular denominators nor plant states are silently clipped.
The original MATLAB implementation is separate; a Python pass is not a MATLAB execution result.